# Simulación del problema bancario con tres cajas independientes

## Introducción

Este notebook resuelve la actividad de teoría de colas del problema bancario. El banco opera durante **480 minutos**, recibe clientes que realizan **retiros (70 %)** o **pagos/consignaciones (30 %)** y dispone inicialmente de tres cajas.

El objetivo es ejecutar múltiples réplicas, medir el desempeño de cada caja y comparar la configuración actual con dos distribuciones exclusivas: **una caja para retiros y dos para pagos**, y **dos cajas para retiros y una para pagos**. También se compara una alternativa de cuatro cajas para evaluar si hace falta ampliar la capacidad.

> **Decisión de modelado:** se implementan tres colas FIFO independientes, coherentes con la indicación del enunciado de tratar las cajas como M/M/1. Cuando varias cajas pueden atender una acción, el cliente elige la que tiene menor carga pendiente. Los clientes dejan de llegar al minuto 480, pero quienes ya llegaron terminan su atención.


## Metodología y supuestos

1. Cada cliente recibe una acción (`retiro` o `pago`) y un tipo (`rapido`, `normal`, `lento`, `muy_lento`) según las probabilidades del enunciado.
2. El tiempo hasta la siguiente llegada y el tiempo de servicio se generan con distribuciones exponenciales cuyas medias dependen de la acción y del tipo.
3. Cada caja conserva su propia cola FIFO. Entre las cajas elegibles se selecciona la de menor carga pendiente; los empates se resuelven aleatoriamente para evitar favorecer siempre la primera caja.
4. Se ejecutan **30 réplicas por escenario** con semillas independientes y reproducibles.
5. Para decidir si se necesita una cuarta caja se usa un criterio explícito de servicio: espera media general mayor que 5 minutos o percentil 95 mayor que 10 minutos. Estos umbrales son supuestos de análisis y pueden modificarse.
6. La mezcla de tipos produce un sistema más general que un M/M/1 homogéneo. Por eso las fórmulas clásicas se usan como referencia conceptual y la validación principal se hace con invariantes, reproducibilidad y Ley de Little observada.


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from math import sqrt
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HORIZONTE_MIN = 8 * 60
N_REPLICAS = 30
SEMILLA_BASE = 202503
UMBRAL_ESPERA_MEDIA = 5.0
UMBRAL_ESPERA_P95 = 10.0

ACCIONES = ("retiro", "pago")
TIPOS = ("rapido", "normal", "lento", "muy_lento")

PROB_ACCION = {"retiro": 0.70, "pago": 0.30}
PROB_TIPO = {
    "retiro": {"rapido": 0.23, "normal": 0.40, "lento": 0.17, "muy_lento": 0.20},
    "pago": {"rapido": 0.10, "normal": 0.20, "lento": 0.30, "muy_lento": 0.40},
}

# Medias (minutos) tomadas de la Tabla 1 del enunciado.
PARAMETROS = {
    "retiro": {
        "rapido": {"servicio": 1.0, "interllegada": 1.0},
        "normal": {"servicio": 2.0, "interllegada": 2.0},
        "lento": {"servicio": 3.0, "interllegada": 3.0},
        "muy_lento": {"servicio": 4.0, "interllegada": 3.0},
    },
    "pago": {
        "rapido": {"servicio": 3.0, "interllegada": 1.0},
        "normal": {"servicio": 3.0, "interllegada": 2.0},
        "lento": {"servicio": 5.0, "interllegada": 3.0},
        "muy_lento": {"servicio": 7.0, "interllegada": 4.0},
    },
}


In [2]:
# Cada escenario define las acciones que puede atender cada caja.
ESCENARIOS = {
    "tres_mixtas": {
        "Caja 1": ("retiro", "pago"),
        "Caja 2": ("retiro", "pago"),
        "Caja 3": ("retiro", "pago"),
    },
    "una_retiros_dos_pagos": {
        "Caja 1": ("retiro",),
        "Caja 2": ("pago",),
        "Caja 3": ("pago",),
    },
    "dos_retiros_un_pago": {
        "Caja 1": ("retiro",),
        "Caja 2": ("retiro",),
        "Caja 3": ("pago",),
    },
    "cuatro_mixtas": {
        "Caja 1": ("retiro", "pago"),
        "Caja 2": ("retiro", "pago"),
        "Caja 3": ("retiro", "pago"),
        "Caja 4": ("retiro", "pago"),
    },
}


def validar_parametros(parametros: dict) -> None:
    '''Falla temprano si las probabilidades o medias no son válidas.'''
    if not np.isclose(sum(PROB_ACCION.values()), 1.0):
        raise ValueError("Las probabilidades de acción deben sumar 1.")

    for accion in ACCIONES:
        if not np.isclose(sum(PROB_TIPO[accion].values()), 1.0):
            raise ValueError(f"Las probabilidades de tipo para {accion} deben sumar 1.")
        for tipo in TIPOS:
            valores = parametros[accion][tipo]
            if valores["servicio"] <= 0 or valores["interllegada"] <= 0:
                raise ValueError("Todas las medias exponenciales deben ser positivas.")

    for nombre, roles in ESCENARIOS.items():
        cubiertas = {accion for acciones in roles.values() for accion in acciones}
        if cubiertas != set(ACCIONES):
            raise ValueError(f"El escenario {nombre} no cubre todas las acciones.")


validar_parametros(PARAMETROS)


## Motor de simulación

El motor genera una secuencia cronológica de clientes y actualiza la disponibilidad de cada caja. La carga pendiente de una caja en el momento de llegada es `max(disponible_en - llegada, 0)`. Esta cantidad resume el trabajo de los clientes que ya están en esa cola.

Se guardan los tiempos de llegada, inicio, fin, servicio, espera y permanencia en el sistema. Esto permite verificar el orden temporal y calcular indicadores sin depender de variables internas ocultas.


In [3]:
@dataclass(frozen=True)
class ClienteGenerado:
    id_cliente: int
    llegada: float
    accion: str
    tipo: str
    tiempo_servicio: float


def _elegir(rng: np.random.Generator, probabilidades: dict[str, float]) -> str:
    opciones = list(probabilidades)
    pesos = np.array([probabilidades[opcion] for opcion in opciones], dtype=float)
    return str(rng.choice(opciones, p=pesos))


def generar_clientes(
    rng: np.random.Generator,
    horizonte: float = HORIZONTE_MIN,
    parametros: dict = PARAMETROS,
) -> list[ClienteGenerado]:
    '''Genera llegadas hasta el cierre usando las medias de la actividad.'''
    clientes: list[ClienteGenerado] = []
    reloj = 0.0
    siguiente_id = 1

    while True:
        accion = _elegir(rng, PROB_ACCION)
        tipo = _elegir(rng, PROB_TIPO[accion])
        media_llegada = parametros[accion][tipo]["interllegada"]
        reloj += float(rng.exponential(media_llegada))
        if reloj > horizonte:
            break

        media_servicio = parametros[accion][tipo]["servicio"]
        clientes.append(
            ClienteGenerado(
                id_cliente=siguiente_id,
                llegada=reloj,
                accion=accion,
                tipo=tipo,
                tiempo_servicio=float(rng.exponential(media_servicio)),
            )
        )
        siguiente_id += 1

    return clientes


def _caja_menos_cargada(
    rng: np.random.Generator,
    accion: str,
    llegada: float,
    disponible_en: dict[str, float],
    roles: dict[str, tuple[str, ...]],
) -> str:
    elegibles = [caja for caja, acciones in roles.items() if accion in acciones]
    if not elegibles:
        raise RuntimeError(f"No existe una caja elegible para {accion}.")

    cargas = {caja: max(disponible_en[caja] - llegada, 0.0) for caja in elegibles}
    carga_minima = min(cargas.values())
    empatadas = [caja for caja, carga in cargas.items() if np.isclose(carga, carga_minima)]
    return str(rng.choice(empatadas))


def simular_replica(
    escenario: str,
    semilla: int,
    horizonte: float = HORIZONTE_MIN,
    parametros: dict = PARAMETROS,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''Simula una jornada y devuelve detalle de clientes y resumen por caja.'''
    validar_parametros(parametros)
    if escenario not in ESCENARIOS:
        raise KeyError(f"Escenario desconocido: {escenario}")

    rng = np.random.default_rng(semilla)
    roles = ESCENARIOS[escenario]
    disponible_en = {caja: 0.0 for caja in roles}
    registros: list[dict] = []

    for cliente in generar_clientes(rng, horizonte, parametros):
        caja = _caja_menos_cargada(
            rng, cliente.accion, cliente.llegada, disponible_en, roles
        )
        inicio = max(cliente.llegada, disponible_en[caja])
        fin = inicio + cliente.tiempo_servicio
        disponible_en[caja] = fin

        registros.append(
            {
                "id_cliente": cliente.id_cliente,
                "escenario": escenario,
                "cajero": caja,
                "accion": cliente.accion,
                "tipo": cliente.tipo,
                "llegada": cliente.llegada,
                "inicio_servicio": inicio,
                "fin_servicio": fin,
                "servicio": cliente.tiempo_servicio,
                "espera": inicio - cliente.llegada,
                "tiempo_sistema": fin - cliente.llegada,
            }
        )

    clientes = pd.DataFrame(registros)
    resumen_filas = []
    for caja in roles:
        grupo = clientes.loc[clientes["cajero"] == caja]
        ocupado_hasta_cierre = sum(
            max(0.0, min(fin, horizonte) - min(max(inicio, 0.0), horizonte))
            for inicio, fin in zip(grupo["inicio_servicio"], grupo["fin_servicio"])
            if inicio < horizonte
        )
        resumen_filas.append(
            {
                "escenario": escenario,
                "cajero": caja,
                "acciones": "/".join(roles[caja]),
                "usuarios_atendidos": len(grupo),
                "atencion_promedio": grupo["servicio"].mean(),
                "espera_promedio": grupo["espera"].mean(),
                "espera_p95": grupo["espera"].quantile(0.95),
                "tiempo_sistema_promedio": grupo["tiempo_sistema"].mean(),
                "utilizacion_hasta_cierre": ocupado_hasta_cierre / horizonte,
                "fin_ultimo_servicio": grupo["fin_servicio"].max(),
            }
        )

    resumen = pd.DataFrame(resumen_filas).fillna(0.0)
    return clientes, resumen


In [4]:
def ejecutar_replicas(
    escenario: str,
    n_replicas: int = N_REPLICAS,
    semilla_base: int = SEMILLA_BASE,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''Ejecuta réplicas con semillas independientes derivadas de SeedSequence.'''
    if n_replicas < 10:
        raise ValueError("La actividad exige ejecutar al menos 10 réplicas.")

    semillas = np.random.SeedSequence(semilla_base).spawn(n_replicas)
    detalles = []
    resumenes = []

    for replica, secuencia in enumerate(semillas, start=1):
        semilla = int(secuencia.generate_state(1, dtype=np.uint32)[0])
        clientes, resumen = simular_replica(escenario, semilla)
        clientes.insert(0, "replica", replica)
        resumen.insert(0, "replica", replica)
        detalles.append(clientes)
        resumenes.append(resumen)

    return pd.concat(detalles, ignore_index=True), pd.concat(resumenes, ignore_index=True)


def intervalo_confianza_95(valores: Iterable[float]) -> pd.Series:
    '''Calcula media e IC95 % normal sin depender de SciPy.'''
    serie = pd.Series(list(valores), dtype=float).dropna()
    media = serie.mean()
    error = 1.96 * serie.std(ddof=1) / sqrt(len(serie))
    return pd.Series({"media": media, "ic95_inferior": media - error, "ic95_superior": media + error})


def validar_replica(clientes: pd.DataFrame, resumen: pd.DataFrame, escenario: str) -> None:
    '''Comprueba conservación, tiempos y elegibilidad de las asignaciones.'''
    if len(clientes) != int(resumen["usuarios_atendidos"].sum()):
        raise AssertionError("La conservación de clientes no se cumple.")
    if (clientes["espera"] < -1e-12).any():
        raise AssertionError("Se detectaron tiempos de espera negativos.")
    if (clientes["inicio_servicio"] < clientes["llegada"] - 1e-12).any():
        raise AssertionError("Un servicio inicia antes de la llegada.")
    if (clientes["fin_servicio"] < clientes["inicio_servicio"] - 1e-12).any():
        raise AssertionError("Un servicio termina antes de iniciar.")

    roles = ESCENARIOS[escenario]
    for fila in clientes.itertuples(index=False):
        if fila.accion not in roles[fila.cajero]:
            raise AssertionError("Un cliente fue enviado a una caja no elegible.")


def validar_little(clientes: pd.DataFrame) -> pd.DataFrame:
    '''Verifica Lq = lambda * Wq por caja usando áreas observadas.'''
    filas = []
    for (replica, escenario, cajero), grupo in clientes.groupby(["replica", "escenario", "cajero"]):
        horizonte_observado = max(float(grupo["fin_servicio"].max()), HORIZONTE_MIN)
        tasa = len(grupo) / horizonte_observado
        wq = float(grupo["espera"].mean())
        lq_por_area = float(grupo["espera"].sum()) / horizonte_observado
        filas.append(
            {
                "replica": replica,
                "escenario": escenario,
                "cajero": cajero,
                "Lq_area": lq_por_area,
                "lambda_Wq": tasa * wq,
                "error_absoluto": abs(lq_por_area - tasa * wq),
            }
        )
    return pd.DataFrame(filas)


## Resultados

Se ejecutan 30 réplicas para cada escenario. Los resultados detallados se guardan en `resultados/` para que puedan auditarse fuera del notebook.


In [5]:
RESULTADOS_DIR = Path("resultados")
RESULTADOS_DIR.mkdir(exist_ok=True)

detalles_por_escenario = {}
resumenes_por_escenario = {}
for nombre_escenario in ESCENARIOS:
    detalle, resumen = ejecutar_replicas(nombre_escenario)
    detalles_por_escenario[nombre_escenario] = detalle
    resumenes_por_escenario[nombre_escenario] = resumen

detalle_total = pd.concat(detalles_por_escenario.values(), ignore_index=True)
resumen_cajeros = pd.concat(resumenes_por_escenario.values(), ignore_index=True)
detalle_total.to_csv(RESULTADOS_DIR / "detalle_clientes.csv", index=False)
resumen_cajeros.to_csv(RESULTADOS_DIR / "resumen_cajeros.csv", index=False)

print(f"Réplicas por escenario: {N_REPLICAS}")
print(f"Escenarios ejecutados: {len(ESCENARIOS)}")
print(f"Registros de clientes simulados: {len(detalle_total):,}")


Réplicas por escenario: 30
Escenarios ejecutados: 4
Registros de clientes simulados: 23,940


### 1. Cajero con menor y mayor tiempo promedio de atención

El tiempo de atención corresponde únicamente al servicio, no a la espera. Se promedia primero dentro de cada réplica y luego entre las 30 réplicas del escenario actual de tres cajas mixtas.


In [6]:
baseline_cajeros = (
    resumen_cajeros.query("escenario == 'tres_mixtas'")
    .groupby("cajero", as_index=False)
    .agg(
        atencion_promedio=("atencion_promedio", "mean"),
        espera_promedio=("espera_promedio", "mean"),
        utilizacion_promedio=("utilizacion_hasta_cierre", "mean"),
        usuarios_promedio=("usuarios_atendidos", "mean"),
    )
)
cajero_menor = baseline_cajeros.loc[baseline_cajeros["atencion_promedio"].idxmin()]
cajero_mayor = baseline_cajeros.loc[baseline_cajeros["atencion_promedio"].idxmax()]

print(baseline_cajeros.round(3).to_string(index=False))
print(f"\nMenor atención promedio: {cajero_menor['cajero']} ({cajero_menor['atencion_promedio']:.3f} min)")
print(f"Mayor atención promedio: {cajero_mayor['cajero']} ({cajero_mayor['atencion_promedio']:.3f} min)")
baseline_cajeros.to_csv(RESULTADOS_DIR / "comparacion_cajeros_baseline.csv", index=False)


cajero  atencion_promedio  espera_promedio  utilizacion_promedio  usuarios_promedio
Caja 1              3.248            0.452                 0.440             66.100
Caja 2              3.317            0.433                 0.441             65.767
Caja 3              3.344            0.518                 0.461             67.633

Menor atención promedio: Caja 1 (3.248 min)
Mayor atención promedio: Caja 3 (3.344 min)


### 2 y 3. Usuarios por tipo y por réplica

La primera tabla muestra el promedio diario de usuarios de cada combinación acción–tipo. La segunda conserva el total de cada réplica. Para cada tipo se identifica la réplica con menor cantidad observada.


In [7]:
baseline = detalles_por_escenario["tres_mixtas"]
conteos_replica_tipo = (
    baseline.groupby(["replica", "accion", "tipo"])
    .size()
    .rename("usuarios")
    .reset_index()
)
promedio_usuarios_tipo = (
    conteos_replica_tipo.groupby(["accion", "tipo"], as_index=False)["usuarios"]
    .mean()
    .rename(columns={"usuarios": "usuarios_promedio_por_dia"})
)
minimos_por_tipo = conteos_replica_tipo.loc[
    conteos_replica_tipo.groupby(["accion", "tipo"])["usuarios"].idxmin()
].sort_values(["accion", "tipo"])

print("Promedio de usuarios por acción y tipo:")
print(promedio_usuarios_tipo.round(2).to_string(index=False))
print("\nRéplica con menor cantidad por acción y tipo:")
print(minimos_por_tipo.to_string(index=False))

conteos_replica_tipo.to_csv(RESULTADOS_DIR / "usuarios_por_tipo_y_replica.csv", index=False)
promedio_usuarios_tipo.to_csv(RESULTADOS_DIR / "promedio_usuarios_por_tipo.csv", index=False)
minimos_por_tipo.to_csv(RESULTADOS_DIR / "minimos_usuarios_por_tipo.csv", index=False)


Promedio de usuarios por acción y tipo:
accion      tipo  usuarios_promedio_por_dia
  pago     lento                      18.63
  pago muy_lento                      24.10
  pago    normal                      13.13
  pago    rapido                       5.97
retiro     lento                      24.93
retiro muy_lento                      26.17
retiro    normal                      54.00
retiro    rapido                      32.57

Réplica con menor cantidad por acción y tipo:
 replica accion      tipo  usuarios
       2   pago     lento         9
       4   pago muy_lento        17
      16   pago    normal         5
      13   pago    rapido         2
       5 retiro     lento        11
      26 retiro muy_lento        16
      12 retiro    normal        42
      22 retiro    rapido        21


### 4 y 5. ¿Hace falta una nueva caja y cómo distribuir cajas exclusivas?

Para cada réplica se calcula la espera media general y el percentil 95. La cuarta caja se recomienda sólo si el escenario actual supera el criterio definido en metodología. Entre las dos distribuciones exclusivas se elige la que obtiene menor espera media; también se informa su diferencia frente a las tres cajas mixtas.


In [8]:
def metricas_generales(detalle: pd.DataFrame) -> pd.DataFrame:
    return (
        detalle.groupby(["escenario", "replica"])
        .agg(
            usuarios=("id_cliente", "size"),
            espera_media=("espera", "mean"),
            espera_p95=("espera", lambda x: x.quantile(0.95)),
            tiempo_sistema_medio=("tiempo_sistema", "mean"),
        )
        .reset_index()
    )

metricas_replica = metricas_generales(detalle_total)
resumen_escenarios = (
    metricas_replica.groupby("escenario", as_index=False)
    .agg(
        usuarios_promedio=("usuarios", "mean"),
        espera_media=("espera_media", "mean"),
        espera_p95=("espera_p95", "mean"),
        tiempo_sistema_medio=("tiempo_sistema_medio", "mean"),
    )
)
ic_espera = (
    metricas_replica.groupby("escenario")["espera_media"]
    .apply(intervalo_confianza_95)
    .unstack()
    .reset_index()
)
resumen_escenarios = resumen_escenarios.merge(ic_espera, on="escenario", how="left")

fila_actual = resumen_escenarios.query("escenario == 'tres_mixtas'").iloc[0]
necesita_nueva_caja = bool(
    fila_actual["espera_media"] > UMBRAL_ESPERA_MEDIA
    or fila_actual["espera_p95"] > UMBRAL_ESPERA_P95
)
exclusivos = resumen_escenarios.query(
    "escenario in ['una_retiros_dos_pagos', 'dos_retiros_un_pago']"
)
mejor_exclusivo = exclusivos.loc[exclusivos["espera_media"].idxmin()]

print(resumen_escenarios.round(3).to_string(index=False))
print(
    f"\n¿Se recomienda una cuarta caja? {'Sí' if necesita_nueva_caja else 'No'} "
    f"(media={fila_actual['espera_media']:.3f}, p95={fila_actual['espera_p95']:.3f} min)."
)
print(
    f"Mejor distribución exclusiva: {mejor_exclusivo['escenario']} "
    f"con espera media de {mejor_exclusivo['espera_media']:.3f} min."
)

metricas_replica.to_csv(RESULTADOS_DIR / "metricas_por_replica.csv", index=False)
resumen_escenarios.to_csv(RESULTADOS_DIR / "comparacion_escenarios.csv", index=False)


            escenario  usuarios_promedio  espera_media  espera_p95  tiempo_sistema_medio  media  ic95_inferior  ic95_superior
        cuatro_mixtas              199.5         0.079       0.343                 3.349  0.079          0.051          0.107
  dos_retiros_un_pago              199.5         4.296      25.166                 7.566  4.296          2.512          6.080
          tres_mixtas              199.5         0.472       3.363                 3.743  0.472          0.351          0.593
una_retiros_dos_pagos              199.5         4.147      19.245                 7.417  4.147          3.536          4.757

¿Se recomienda una cuarta caja? No (media=0.472, p95=3.363 min).
Mejor distribución exclusiva: una_retiros_dos_pagos con espera media de 4.147 min.


In [9]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
orden = resumen_escenarios.sort_values("espera_media")["escenario"]

resumen_escenarios.set_index("escenario").loc[orden, "espera_media"].plot.bar(
    ax=axes[0], color="#2F6690", title="Espera media por escenario"
)
resumen_escenarios.set_index("escenario").loc[orden, "espera_p95"].plot.bar(
    ax=axes[1], color="#D1495B", title="Percentil 95 de espera"
)
(
    resumen_cajeros.groupby("escenario")["utilizacion_hasta_cierre"].mean().loc[orden]
).plot.bar(ax=axes[2], color="#3A7D44", title="Utilización media de cajas")

for ax in axes:
    ax.set_xlabel("")
    ax.set_ylabel("minutos" if ax is not axes[2] else "proporción")
    ax.tick_params(axis="x", rotation=35)
    ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(RESULTADOS_DIR / "comparacion_escenarios.png", dpi=160, bbox_inches="tight")
plt.close(fig)
print("Gráfico guardado en resultados/comparacion_escenarios.png")


Gráfico guardado en resultados/comparacion_escenarios.png


## Validación del modelo

La validación combina conservación de clientes, coherencia temporal, elegibilidad de las cajas, reproducibilidad, frecuencias observadas y la Ley de Little (`Lq = λWq`).


In [10]:
for escenario, detalle in detalles_por_escenario.items():
    resumen = resumenes_por_escenario[escenario]
    for replica in range(1, N_REPLICAS + 1):
        validar_replica(
            detalle.loc[detalle["replica"] == replica],
            resumen.loc[resumen["replica"] == replica],
            escenario,
        )

repro_a, repro_resumen_a = simular_replica("tres_mixtas", semilla=98765)
repro_b, repro_resumen_b = simular_replica("tres_mixtas", semilla=98765)
pd.testing.assert_frame_equal(repro_a, repro_b)
pd.testing.assert_frame_equal(repro_resumen_a, repro_resumen_b)

frecuencia_accion = baseline["accion"].value_counts(normalize=True)
assert abs(frecuencia_accion["retiro"] - PROB_ACCION["retiro"]) < 0.03
for accion in ACCIONES:
    observadas = baseline.loc[baseline["accion"] == accion, "tipo"].value_counts(normalize=True)
    for tipo in TIPOS:
        assert abs(observadas[tipo] - PROB_TIPO[accion][tipo]) < 0.04

little = validar_little(detalle_total)
assert little["error_absoluto"].max() < 1e-10
little.to_csv(RESULTADOS_DIR / "validacion_little.csv", index=False)

print("Todas las validaciones finalizaron correctamente.")
print("Frecuencias observadas de acción:")
print(frecuencia_accion.round(4).to_string())
print(f"Error máximo en Ley de Little: {little['error_absoluto'].max():.3e}")


Todas las validaciones finalizaron correctamente.
Frecuencias observadas de acción:
accion
retiro    0.6901
pago      0.3099
Error máximo en Ley de Little: 8.882e-16


## Conclusiones

Las conclusiones se construyen con los resultados calculados. Los umbrales de calidad son supuestos modificables y la mezcla de tipos hace que el sistema no sea un M/M/1 homogéneo puro.


In [11]:
comparacion_cuarta = resumen_escenarios.query("escenario == 'cuatro_mixtas'").iloc[0]
mejora_cuarta = 100 * (
    fila_actual["espera_media"] - comparacion_cuarta["espera_media"]
) / max(fila_actual["espera_media"], 1e-12)
diferencia_exclusivo = mejor_exclusivo["espera_media"] - fila_actual["espera_media"]

print("CONCLUSIONES")
print(
    f"1. Las tres cajas mixtas presentan una espera media de {fila_actual['espera_media']:.3f} min "
    f"y un p95 de {fila_actual['espera_p95']:.3f} min."
)
print(
    f"2. Con el criterio de {UMBRAL_ESPERA_MEDIA:.1f} min de media o "
    f"{UMBRAL_ESPERA_P95:.1f} min de p95, "
    f"{'se recomienda' if necesita_nueva_caja else 'no se recomienda'} crear una cuarta caja. "
    f"La alternativa de cuatro cajas reduce la espera media en {mejora_cuarta:.1f} %."
)
print(
    f"3. Si el banco exige cajas exclusivas, la mejor distribución simulada es "
    f"{mejor_exclusivo['escenario']}; su diferencia frente a las cajas mixtas es "
    f"{diferencia_exclusivo:+.3f} min de espera media."
)
print(
    "4. La recomendación debe revisarse si cambian las probabilidades, los tiempos o los umbrales; "
    "todos están concentrados al inicio del notebook para facilitar nuevos experimentos."
)


CONCLUSIONES
1. Las tres cajas mixtas presentan una espera media de 0.472 min y un p95 de 3.363 min.
2. Con el criterio de 5.0 min de media o 10.0 min de p95, no se recomienda crear una cuarta caja. La alternativa de cuatro cajas reduce la espera media en 83.3 %.
3. Si el banco exige cajas exclusivas, la mejor distribución simulada es una_retiros_dos_pagos; su diferencia frente a las cajas mixtas es +3.675 min de espera media.
4. La recomendación debe revisarse si cambian las probabilidades, los tiempos o los umbrales; todos están concentrados al inicio del notebook para facilitar nuevos experimentos.
